# 12 — Bulk deconvolution: the IAF fraction of fibroblasts in pretreatment biopsies

Notebooks 09/10 scored the IAF *program* in bulk (signature means). This notebook
asks the compositional question the program score cannot answer: **what fraction
of the fibroblast compartment is in the IAF state**, estimated by deconvolution
of pretreatment bulk biopsies against a single-cell reference — and whether that
fraction, not just the program level, tracks anti-TNF response.

Design brief: `Bulk_IBD_scRNA_Integration_Brief_2026-11-23.md` (locked 2026-11-23).
Method order is fixed by the brief: reference → pseudo-bulk validation gate →
deconvolution → pre-registered primary model. The gate decides the method's
reach; nothing downstream is tuned after seeing response labels.


## Pre-registration (locked before any deconvolution is run)

**Primary variable.** IAF fraction of fibroblasts, `IAF_frac` = deconvolved IAF
component / (IAF + non-IAF fibroblast components), per pretreatment sample.

**Primary endpoint.** Association of `IAF_frac` with anti-TNF response (R vs NR)
in pretreatment bulk biopsies.

**Primary model.** Per cohort/stratum logistic regression
`response ~ IAF_frac + INFLAM6`, where INFLAM6 (S100A8, S100A9, IL1B, CXCL8,
LCN2, REG1A; per-gene z, mean — the nb09/nb10 convention) is the pre-declared
inflammation-burden covariate (the severity adjustment of nb09/nb10, retained).
The per-stratum log-ORs for `IAF_frac` are combined by DerSimonian–Laird
random-effects meta-analysis, mirroring nb10. Secondary: AUC of `IAF_frac`
alone; ΔAUC over the INFLAM6-only model. Clinical severity (e.g. Mayo) enters
only as a sensitivity analysis where it is recorded.

**Validation gate (decides everything downstream).** Pseudo-bulks are built
from held-out single-cell donors with known IAF-of-fibroblast ratios. The
deconvolution passes if, across pseudo-bulks, Spearman rho(estimated, true)
>= 0.80 AND median absolute error <= 10 percentage points. **If the gate
fails, the pre-registered downgrade applies**: no IAF-fraction claims; report
the fibroblast-total fraction and the nb09 signature scores only.

**Cohorts.** GSE12251, GSE16879 (UC and CD reported as separate strata, as in
nb09), GSE23597, GSE73661 — plus GSE100833, the third cohort of the Friedrich
et al. 2021 validation set (GSE16879, GSE12251, GSE100833), not previously run
here. GSE100833 is curated first (disease, tissue, platform, endpoint) and is
analysed as its own stratum; strata are never pooled raw.

**Reference.** Smillie et al. 2019 (UC colon) full-atlas compartments plus
fibroblast resolution, with Martin et al. 2019 (CD ileum) fibroblasts added.
IAF labels come from the frozen ruler (`ruler_tool/frozen_ruler.py`, AUCell,
T = 0.113171) — the same calls as the manuscript, not a re-derivation.


In [ ]:
import warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np, pandas as pd
from scipy import stats
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

DATA = Path.home() / "Documents/immune-endotypes/data"
BULK = DATA / "bulk_response"; BULK.mkdir(parents=True, exist_ok=True)
MOD  = Path.home() / "Documents/github/immune-endotypes-portfolio/modules/05_cross_cohort"
import sys; sys.path.insert(0, str(MOD / "ruler_tool"))
from frozen_ruler import score_iaf, SIG9_HUMAN

INFLAM6 = ["S100A8","S100A9","IL1B","CXCL8","LCN2","REG1A"]
rng = np.random.default_rng(20261123)
print("paths ok | MOD:", MOD.exists(), "| ruler imported")


## 1 — Reference: compartments + fibroblast resolution

Groups: `IAF`, `Fibroblast_other`, and the major non-fibroblast compartments
from the Smillie atlas annotations (epithelial, myeloid, T/NK, B/plasma,
endothelial, other stromal). Martin fibroblasts are appended to the two
fibroblast groups so the fibroblast reference spans UC colon + CD ileum.
Reference profiles = mean normalized expression per group (linear space; see Amendment 2); markers are
selected per group in section 2.


In [ ]:
import scanpy as sc, anndata as ad

sm = sc.read_h5ad(DATA / "smillie_uc/adata_qc.h5ad")
print("Smillie:", sm.shape, "| obs:", list(sm.obs.columns))
# Fibroblast compartment = the authors' eight published fibroblast clusters
# (the frozen ruler's calibration compartment; see ruler_tool README).
print(sm.obs["Cluster"].value_counts().head(30))


In [ ]:
# --- build the reference object -------------------------------------------
# Smillie fibroblast compartment = the authors' eight published fibroblast
# clusters (the frozen ruler's calibration compartment, step1 definition;
# the WNT2B+/WNT5B+/RSPO3+ subsets are fibroblasts named by markers):
FIB_CLUSTERS = ["WNT2B+ Fos-lo 1", "WNT2B+ Fos-lo 2", "WNT2B+ Fos-hi",
                "WNT5B+ 1", "WNT5B+ 2", "RSPO3+",
                "Inflammatory Fibroblasts", "Myofibroblasts"]
missing = [c for c in FIB_CLUSTERS if c not in set(sm.obs["Cluster"].astype(str))]
assert not missing, f"fibroblast clusters missing from atlas: {missing}"
sm_fib = sm[sm.obs["Cluster"].astype(str).isin(FIB_CLUSTERS)].copy()
out = score_iaf(sm_fib, species="human")
sm_fib.obs["ref_group"] = np.where(out["iaf_call"].values, "IAF", "Fibroblast_other")
print("Smillie fibroblast reference:", sm_fib.n_obs, "cells |",
      sm_fib.obs["ref_group"].value_counts().to_dict())

# Non-fibroblast compartments from Cluster names (explicit rules; the printed
# mapping is the record - eyeball it before proceeding)
def compartment(cluster):
    c = str(cluster)
    if any(k in c for k in ["Plasma", "Follicular"]) or c.startswith("B "): return "B_plasma"
    if any(k in c for k in ["CD4", "CD8", "Treg", "NK", "ILC"]): return "T_NK"
    if any(k in c for k in ["Macrophage", "DC", "Mast", "Monocyte", "Neutrophil"]): return "Myeloid"
    if any(k in c for k in ["Endothelial", "Venule", "Microvascular", "Lymphatic"]): return "Endothelial"
    if any(k in c for k in ["Enterocyte", "TA", "Goblet", "Stem", "Best4", "Tuft",
                            "Enteroendocrine", "Secretory", "Cycling"]): return "Epithelial"
    return "Other"
sm_other = sm[~sm.obs["Cluster"].astype(str).isin(FIB_CLUSTERS)].copy()
sm_other.obs["ref_group"] = sm_other.obs["Cluster"].map(compartment)
print(sm_other.obs.groupby("ref_group")["Cluster"].unique())
print("compartments:", sm_other.obs["ref_group"].value_counts().to_dict())


In [ ]:
# Martin fibroblasts appended to the fibroblast groups (nb03/nb04 objects)
# Martin fibroblast object as saved by nb04 (the fibroblast-subclustering input)
ma_fib = sc.read_h5ad(DATA / "martin_2019/adata_martin_fibroblast.h5ad")
print("Martin fibroblasts:", ma_fib.shape, "| obs:", list(ma_fib.obs.columns))
out_m = score_iaf(ma_fib, species="human")
ma_fib.obs["ref_group"] = np.where(out_m["iaf_call"].values, "IAF", "Fibroblast_other")
print("Martin fibroblast reference:", ma_fib.obs["ref_group"].value_counts().to_dict())
print("NOTE: if the Martin fibroblast selection above does not match nb04's,")
print("stop and align it before proceeding — the reference defines the estimand.")


In [ ]:
# --- donor split: reference donors vs held-out donors (for the gate) -------
def donor_col(obs):
    for c in ["Subject","subject","donor","Donor","patient","Patient","sample_id"]:
        if c in obs.columns: return c
    return None

def lognorm(a):
    a = a.copy()
    sc.pp.normalize_total(a, target_sum=1e4); sc.pp.log1p(a)
    return a

sm_fib = lognorm(sm_fib); ma_fib = lognorm(ma_fib); sm_other = lognorm(sm_other)
for a in (sm_fib, ma_fib):
    dc = donor_col(a.obs)
    donors = a.obs[dc].astype(str).unique()
    rng.shuffle(donors)
    a.obs["donor_split"] = np.where(a.obs[dc].astype(str).isin(donors[:max(1,int(.7*len(donors)))]),
                                    "reference", "heldout")
    print("donor split:", a.obs["donor_split"].value_counts().to_dict(), "| donor col:", dc)


## 2 — Deconvolution: DWLS in pure numpy

Dampened weighted least squares (Tsoucas et al. 2019), implemented here so the
notebook is self-contained in the Python environment. Markers: per-group
one-vs-rest differential expression on the reference-donor cells (top 60 per
group by logFC among genes detected in >=10% of group cells). The gate in
section 3 — not preference — decides whether DWLS estimates are used as-is.


In [ ]:
from scipy.optimize import nnls

def harmonize(objs):
    common = objs[0].var_names
    for a in objs[1:]:
        common = common.intersection(a.var_names)
    print("common genes across reference objects:", len(common))
    return [a[:, common].copy() for a in objs]

def group_profiles(objs, group_col="ref_group"):
    # per-group mean expression, computed sparsely object by object -
    # the full stack is never densified (350k cells x 15k genes would OOM)
    genes = objs[0].var_names
    sums, counts = {}, {}
    for a in objs:
        g = a.obs[group_col].astype(str)
        for grp in g.unique():
            # linear space: the reference objects were log-normalized in the
            # donor-split cell; DWLS/MuSiC-style references are means of
            # normalized counts, so undo log1p per stored entry before summing
            Xg = a.X[g.values == grp].tocsr().copy()
            Xg.data = np.expm1(Xg.data)
            sums[grp] = sums.get(grp, 0) + np.asarray(Xg.sum(axis=0)).ravel()
            counts[grp] = counts.get(grp, 0) + int((g.values == grp).sum())
    return pd.DataFrame({grp: sums[grp] / counts[grp] for grp in sorted(sums)}, index=genes)

def detection_fractions(objs, group_col="ref_group"):
    genes = objs[0].var_names
    hits, counts = {}, {}
    for a in objs:
        g = a.obs[group_col].astype(str)
        for grp in g.unique():
            Xg = a.X[g.values == grp]
            hits[grp] = hits.get(grp, 0) + np.asarray((Xg > 0).sum(axis=0)).ravel()
            counts[grp] = counts.get(grp, 0) + int((g.values == grp).sum())
    return pd.DataFrame({grp: hits[grp] / counts[grp] for grp in sorted(hits)}, index=genes)

def select_markers(prof, detfrac, topn=60, min_frac=0.10):
    markers = {}
    for grp in prof.columns:
        lfc = np.log2((prof[grp] + 0.01) / (prof.drop(columns=[grp]).max(axis=1) + 0.01))
        cand = lfc[detfrac[grp] >= min_frac]
        markers[grp] = list(cand.sort_values(ascending=False).head(topn).index)
    return markers

def dwls(bulk_vec, S, iters=200, damp=0.1):
    # S: genes x groups signature matrix (marker rows); bulk_vec aligned to S.index
    x, _ = nnls(S.values, bulk_vec)
    if x.sum() == 0: return pd.Series(0.0, index=S.columns)
    x = x / x.sum()
    for _ in range(iters):
        w = 1.0 / (S.values @ x + damp)
        Sw = S.values * np.sqrt(w)[:, None]; bw = bulk_vec * np.sqrt(w)
        xn, _ = nnls(Sw, bw)
        if xn.sum() == 0: break
        xn = xn / xn.sum()
        if np.max(np.abs(xn - x)) < 1e-5: x = xn; break
        x = xn
    return pd.Series(x, index=S.columns)

print("dwls machinery defined")


In [ ]:
# reference profiles from REFERENCE donors only; markers on the same cells.
# Smillie and Martin were deposited with different gene universes, so the
# objects are harmonized to their common genes before any stacking.
ref_objs_raw = [sm_fib[sm_fib.obs["donor_split"] == "reference"],
                ma_fib[ma_fib.obs["donor_split"] == "reference"], sm_other]
ref_objs = harmonize(ref_objs_raw)
prof = group_profiles(ref_objs)
detfrac = detection_fractions(ref_objs)
markers = select_markers(prof, detfrac)
all_markers = sorted(set(sum(markers.values(), [])))
S_ref = prof.loc[all_markers]
print("reference groups:", list(prof.columns))
print("markers:", {k: len(v) for k, v in markers.items()}, "| union:", len(all_markers))
S_ref.to_csv(MOD / "results/nb12_reference_signatures.csv")


## 3 — The gate: pseudo-bulk validation on held-out donors

**Amendment (2026-11-24, recorded before any response labels are touched).**
The first implementation drew fibroblasts at random from the held-out pools,
so the true IAF fraction had almost no dynamic range (Smillie 4.3–5.7%;
Martin constant at 0.57% — its held-out pool holds 2 IAF cells), and DWLS
returned an IAF component of exactly 0 throughout: a rank-based gate on a
near-constant truth can neither pass nor fail meaningfully. The construction
below replaces random draws with **titration**: pseudo-bulks are built at
known IAF fractions spanning 0–50% of the fibroblast compartment. The pass
criteria are unchanged from the pre-registration (rho >= 0.80, median |err|
<= 10 pp). Per the brief ("the pseudo-bulk validation decides the final
method"), both plain NNLS and DWLS are evaluated and the passing solver is
selected; if neither passes, the pre-registered downgrade applies. Martin
cannot be titrated from its held-out pool (2 IAF cells), so the gate runs on
Smillie held-out donors; Martin contributes to the reference only.

**Amendment 2 (2026-11-24).** The titration gate's first run failed for both
solvers (IAF estimated at ~0 even at 50% true fraction). The reference
profiles in that run were means of log-normalized values (mean-of-logs),
while the pseudo-bulks were log-of-sums — a nonstandard, internally
inconsistent formulation that systematically buries sparse marker genes.
The standard DWLS/MuSiC construction — reference profiles as means of
normalized counts, pseudo-bulk and bulk in linear space — is therefore run
as the final method variant under the brief's validation clause. Pass
criteria unchanged. If this variant also fails, the downgrade is final and
the failure itself is the reported result.


In [ ]:
def titrated_pseudobulk(fib_a, other_a, target_frac, n_fib=800, n_other=3000, seed=0):
    # fibroblast compartment titrated to a KNOWN IAF fraction; background of
    # non-fibroblast cells added; sums aligned by gene name across cohorts
    r = np.random.default_rng(seed)
    iaf_idx = np.where(fib_a.obs["ref_group"].values == "IAF")[0]
    oth_idx = np.where(fib_a.obs["ref_group"].values == "Fibroblast_other")[0]
    n_iaf = int(round(target_frac * n_fib)); n_oth = n_fib - n_iaf
    fi = np.concatenate([
        r.choice(iaf_idx, size=n_iaf, replace=len(iaf_idx) < n_iaf),
        r.choice(oth_idx, size=n_oth, replace=len(oth_idx) < n_oth)])
    oi = r.choice(other_a.n_obs, size=min(n_other, other_a.n_obs), replace=False)
    Xf = fib_a.layers["counts"] if "counts" in fib_a.layers else fib_a.X
    Xo = other_a.layers["counts"] if "counts" in other_a.layers else other_a.X
    sf = pd.Series(np.asarray(Xf[fi].sum(axis=0)).ravel(), index=fib_a.var_names)
    so = pd.Series(np.asarray(Xo[oi].sum(axis=0)).ravel(), index=other_a.var_names)
    bulk = sf.add(so, fill_value=0)
    bulk = bulk / bulk.sum() * 1e4   # linear CPM - same space as the reference
    return bulk, float(fib_a.obs["ref_group"].iloc[fi].eq("IAF").mean())

def nnls_plain(bulk_vec, S):
    x, _ = nnls(S.values, bulk_vec)
    return pd.Series(x / x.sum() if x.sum() > 0 else x, index=S.columns)

LEVELS = [0.0, 0.05, 0.10, 0.20, 0.35, 0.50]
sm_ho = sm_fib[sm_fib.obs["donor_split"] == "heldout"]
rows = []
for f in LEVELS:
    for s in range(4):
        bv, true_f = titrated_pseudobulk(sm_ho, sm_other, f, seed=int(f * 1000) + s)
        for solver_name, solver in [("dwls", dwls), ("nnls", nnls_plain)]:
            est = solver(bv.reindex(S_ref.index).fillna(0).values, S_ref)
            denom = est["IAF"] + est["Fibroblast_other"]
            rows.append({"solver": solver_name, "target": f, "seed": s,
                         "true_iaf_frac": true_f,
                         "est_iaf_frac": float(est["IAF"] / denom) if denom > 0 else 0.0})
gate = pd.DataFrame(rows)
verdicts = {}
for solver_name, gsub in gate.groupby("solver"):
    rho = stats.spearmanr(gsub.true_iaf_frac, gsub.est_iaf_frac).statistic
    mae = float((gsub.est_iaf_frac - gsub.true_iaf_frac).abs().median())
    verdicts[solver_name] = bool(rho >= 0.80 and mae <= 0.10)
    print(f"{solver_name}: rho = {rho:.3f} | median |err| = {mae*100:.1f} pp | PASS = {verdicts[solver_name]}")
SOLVER = next((k for k, v in verdicts.items() if v), None)
GATE_PASS = SOLVER is not None
deconv = dwls if SOLVER == "dwls" else nnls_plain
print("selected solver:", SOLVER, "| GATE_PASS =", GATE_PASS)
gate.to_csv(MOD / "results/nb12_pseudobulk_gate.csv", index=False)


## 4 — Bulk cohorts: download, gene-level matrices, INFLAM6

Series matrices are probe-level (GPL570 / GPL6244). Probe→gene collapse uses
the GEO platform annotation (max-mean probe per gene) — genome-wide, unlike
nb09's signature-only probe map. INFLAM6 is computed with the nb09 convention
(per-gene z within cohort, mean over available genes).


In [ ]:
import gzip, io, subprocess, json

URLS = {
 "GSE12251": "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE12nnn/GSE12251/matrix/GSE12251_series_matrix.txt.gz",
 "GSE16879": "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE16nnn/GSE16879/matrix/GSE16879_series_matrix.txt.gz",
 "GSE23597": "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE23nnn/GSE23597/matrix/GSE23597_series_matrix.txt.gz",
 "GSE73661": "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE73nnn/GSE73661/matrix/GSE73661_series_matrix.txt.gz",
 "GSE100833":"https://ftp.ncbi.nlm.nih.gov/geo/series/GSE100nnn/GSE100833/matrix/GSE100833_series_matrix.txt.gz",
}
for gse, url in URLS.items():
    out = BULK / f"{gse}_series_matrix.txt.gz"
    if not out.exists():
        subprocess.run(["curl","-sSL","-o",str(out),url], check=True)
    print(gse, "ok", out.stat().st_size//1024, "KB")

def load_series(path):
    meta = {"title": [], "char": [], "platform": None}
    tbl = []
    with gzip.open(path, "rt", errors="replace") as f:
        for line in f:
            if line.startswith("!series_matrix_table_begin"): tbl.append("BEGIN"); continue
            if line.startswith("!series_matrix_table_end"): break
            if tbl and tbl[0] == "BEGIN": tbl.append(line); continue
            if line.startswith("!Sample_title"):
                meta["title"] = [v.strip('"') for v in line.rstrip("\n").split("\t")[1:]]
            elif line.startswith("!Sample_characteristics"):
                meta["char"].append([v.strip('"') for v in line.rstrip("\n").split("\t")[1:]])
            elif line.startswith("!Sample_platform_id"):
                meta["platform"] = line.rstrip("\n").split("\t")[1].strip('"')
    df = pd.read_csv(io.StringIO("".join(tbl[1:])), sep="\t", index_col=0)
    df.index = df.index.astype(str).str.strip('"'); df.columns = [c.strip('"') for c in df.columns]
    return df, meta

def char_series(chars, key):
    for row in chars:
        if row and row[0].lower().startswith(key.lower()):
            return [v.split(":",1)[1].strip() if ":" in v else v for v in row]
    return [""] * len(chars[0]) if chars else []
print("loaders defined")


In [ ]:
# probe -> gene maps from GEO platform annotations (cached)
def platform_gene_map(platform):
    cache = BULK / f"{platform}_probe_gene.csv"
    if cache.exists(): return pd.read_csv(cache, index_col=0)["gene"].to_dict()
    url = f"https://ftp.ncbi.nlm.nih.gov/geo/platforms/{platform[:-3]}nnn/{platform}/annot/{platform}.annot.gz"
    raw = subprocess.run(["curl","-sSL",url], capture_output=True, check=True).stdout
    txt = gzip.decompress(raw).decode("utf-8", "replace")
    lines = txt.splitlines()
    i0 = next(i for i, l in enumerate(lines) if l.startswith("!platform_table_begin")) + 1
    i1 = next(i for i, l in enumerate(lines) if l.startswith("!platform_table_end"))
    t = pd.read_csv(io.StringIO("\n".join(lines[i0:i1])), sep="\t", dtype=str)
    sym_col = [c for c in t.columns if "Gene symbol" in c][0]
    m = {}
    for pid, sym in zip(t["ID"].astype(str), t[sym_col].astype(str)):
        s = sym.split("///")[0].strip()
        if s and s != "---": m[pid] = s
    pd.Series(m).rename("gene").to_csv(cache)
    return m

def gene_matrix(df, platform):
    m = platform_gene_map(platform)
    g = df.rename(index=m)
    g = g[g.index.notna()]
    return g.groupby(level=0).mean()

def inflam6(gm):
    z = (gm - gm.mean(axis=1).values[:,None]) / gm.std(axis=1).replace(0, np.nan).values[:,None]
    return z.reindex([g for g in INFLAM6 if g in z.index]).mean(axis=0)
print("gene-matrix + INFLAM6 helpers defined")


## 5 — GSE100833 curation (before any analysis of it)

Per the brief's stratification rule: record disease, tissue, platform and the
response-endpoint definition from the series itself. If the endpoint is not a
pretreatment anti-TNF response label, GSE100833 is reported descriptively
only and excluded from the primary meta-analysis.


In [ ]:
df100, meta100 = load_series(BULK / "GSE100833_series_matrix.txt.gz")
print("platform:", meta100["platform"], "| n samples:", len(meta100["title"]))
for row in meta100["char"]:
    print("CHAR:", row[0], "| values:", pd.Series(row).value_counts().to_dict())
print("titles sample:", meta100["title"][:6])
# >>> Fill the curation record from what is printed above, then set the flag.
curation = {"gse": "GSE100833", "platform": meta100["platform"],
            "n_samples": len(meta100["title"]),
            "disease": "FILL_FROM_CHARS", "tissue": "FILL_FROM_CHARS",
            "endpoint": "FILL_FROM_CHARS",
            "pretreatment_response_label": "FILL True/False"}
pd.DataFrame([curation]).to_csv(MOD / "results/nb12_gse100833_curation.csv", index=False)
print("curation template saved — complete it before running section 6 for this cohort")


## 6 — Deconvolution + pre-registered primary model

Response labels follow nb09's per-cohort parsing (same characteristic fields).
If `GATE_PASS` is False, this section stops at the downgrade: fibroblast-total
fractions and a pointer back to nb09/nb10 scores.


In [ ]:
def irls_logit(X, y, ridge=1e-6):
    X = np.column_stack([np.ones(len(X)), X]); beta = np.zeros(X.shape[1])
    for _ in range(100):
        eta = X @ beta; p = 1/(1+np.exp(-eta)); W = p*(1-p) + 1e-9
        H = X.T @ (X * W[:,None]) + ridge*np.eye(X.shape[1])
        step = np.linalg.solve(H, X.T @ (y - p) - ridge*beta)
        beta = beta + step
        if np.max(np.abs(step)) < 1e-8: break
    cov = np.linalg.inv(H); se = np.sqrt(np.diag(cov))
    return beta, se

def auc(y, s):
    from sklearn.metrics import roc_auc_score
    return float(roc_auc_score(y, s)) if len(set(y)) == 2 else np.nan

print("model helpers defined | GATE_PASS =", GATE_PASS)


In [ ]:
# --- per-cohort labels (nb09 parsing) + deconvolution ----------------------
COHORTS = {}
df, meta = load_series(BULK / "GSE12251_series_matrix.txt.gz")
resp = char_series(meta["char"], "wk8rsphm")
COHORTS["GSE12251_UC"] = (df, meta["platform"],
    pd.Series({df.columns[i]: ("R" if resp[i] == "Yes" else "NR") for i in range(len(df.columns))}))
df, meta = load_series(BULK / "GSE16879_series_matrix.txt.gz")
dis  = char_series(meta["char"], "disease"); tim = char_series(meta["char"], "before or after first infliximab treatment")
resp = char_series(meta["char"], "response to infliximab")
for strat in ["UC","CD"]:
    lab = {df.columns[i]: ("R" if resp[i] == "Yes" else "NR") for i in range(len(df.columns))
           if tim[i].startswith("Before") and resp[i] in ("Yes","No") and dis[i] == strat}
    COHORTS[f"GSE16879_{strat}"] = (df, meta["platform"], pd.Series(lab))
# GSE23597 / GSE73661 / GSE100833: label fields differ per series — parse here
# following the same pattern after inspecting each series' characteristics
# (print them first; do not guess field names).
for gse in ["GSE23597","GSE73661","GSE100833"]:
    _df, _meta = load_series(BULK / f"{gse}_series_matrix.txt.gz")
    print(gse, "platform:", _meta["platform"])
    for row in _meta["char"]: print("  CHAR:", row[0], "|", pd.Series(row).value_counts().to_dict())
print("labelled strata:", list(COHORTS))


In [ ]:
results, frac_rows = [], []
for name, (df, platform, labels) in COHORTS.items():
    gm_log = gene_matrix(df, platform)
    infl = inflam6(gm_log)   # nb09 convention: INFLAM6 on the log-scale matrix
    gm = np.power(2.0, gm_log) if gm_log.values.max() < 30 else gm_log
    common = S_ref.index.intersection(gm.index)
    print(name, "| genes shared with reference markers:", len(common), "/", len(S_ref))
    est = pd.DataFrame({s: deconv(gm.loc[common, s].values, S_ref.loc[common])
                        for s in gm.columns}).T
    est["IAF_frac"] = est["IAF"] / (est["IAF"] + est["Fibroblast_other"]).replace(0, np.nan)
    est["FIB_total"] = est["IAF"] + est["Fibroblast_other"]
    est["INFLAM6"] = infl
    est["label"] = labels
    est = est.dropna(subset=["label"])
    est["stratum"] = name
    frac_rows.append(est.reset_index(names="sample"))
    if not GATE_PASS:
        results.append({"stratum": name, "note": "gate failed — downgrade: FIB_total only"})
        continue
    y = (est["label"] == "R").astype(float).values
    X = np.column_stack([est["IAF_frac"].values, est["INFLAM6"].values])
    beta, se = irls_logit(X, y)
    z = beta[1] / se[1]
    results.append({"stratum": name, "n": len(est), "n_R": int(y.sum()),
                    "logOR_IAFfrac": beta[1], "se": se[1],
                    "p": float(2*stats.norm.sf(abs(z))),
                    "AUC_IAFfrac": auc(y, est["IAF_frac"].values),
                    "AUC_INFLAM6": auc(y, est["INFLAM6"].values)})
    print(name, "done")

frac = pd.concat(frac_rows)
frac.to_csv(MOD / "results/nb12_deconv_fractions.csv", index=False)
res = pd.DataFrame(results)
print(res.to_string(index=False))


In [ ]:
# --- random-effects meta-analysis of the IAF_frac log-OR (DerSimonian-Laird)
if GATE_PASS and "logOR_IAFfrac" in res.columns:
    r = res.dropna(subset=["logOR_IAFfrac"])
    yi, vi = r["logOR_IAFfrac"].values, r["se"].values**2
    Q = float(np.sum((yi - np.average(yi, weights=1/vi))**2 / vi))
    dfq = len(yi) - 1
    tau2 = max(0.0, (Q - dfq) / (np.sum(1/vi) - np.sum(1/vi**2)/np.sum(1/vi)))
    w = 1/(vi + tau2); mu = float(np.average(yi, weights=w)); se_mu = float(np.sqrt(1/np.sum(w)))
    meta_row = {"stratum": "META_random_effects", "logOR_IAFfrac": mu, "se": se_mu,
                "p": float(2*stats.norm.sf(abs(mu/se_mu))), "tau2": tau2, "Q": Q}
    res = pd.concat([res, pd.DataFrame([meta_row])], ignore_index=True)
    print(f"META: logOR = {mu:.3f} (SE {se_mu:.3f}), OR = {np.exp(mu):.2f}, p = {meta_row['p']:.2g}, tau2 = {tau2:.3f}")
res.to_csv(MOD / "results/nb12_primary_results.csv", index=False)

if GATE_PASS and "logOR_IAFfrac" in res.columns:
    r = res[res.stratum != "META_random_effects"].dropna(subset=["logOR_IAFfrac"])
    fig, ax = plt.subplots(figsize=(6, 3.2))
    ax.errorbar(r["logOR_IAFfrac"], range(len(r)), xerr=1.96*r["se"], fmt="o", capsize=3)
    ax.axvline(0, color="grey", lw=1)
    ax.set_yticks(range(len(r))); ax.set_yticklabels(r["stratum"])
    ax.set_xlabel("log-OR of response per unit IAF_frac (adj. INFLAM6)")
    fig.tight_layout(); fig.savefig(MOD / "figures/nb12_forest.png", dpi=200)
    print("forest plot saved")


## Interpretation (filled 2026-11-24, after the run)

- **Gate outcome: FAILED for both solvers, under both formulations.**
  Log-space run: IAF estimated at 0 at every titration level. Linear-space
  run (Amendment 2): DWLS tracks monotonically but with severe compression
  (true 0.35 -> ~0.06-0.17; true 0.50 -> ~0.20-0.24; true <= 0.20 -> 0;
  median |err| ~17 pp); NNLS shows a near-constant positive offset
  (~0.15-0.33 at every level, including ~0.30 at true 0). Neither meets the
  pre-registered rule (rho >= 0.80 AND median |err| <= 10 pp).
- **Consequence (pre-registered downgrade, applied): no IAF-fraction
  claims.** The primary endpoint is not estimable from bulk with this
  reference and method family. This is a measurement result, not a tuning
  failure: the state that is unambiguous at single-cell resolution does not
  survive transfer to bulk at the level of composition, even when it makes
  up half the fibroblast compartment in silico.
- **Downgrade-level reporting (descriptive only):** the deconvolved
  fibroblast-total fraction is lower in responders in all three wired
  strata (GSE12251_UC median 0.177 R vs 0.275 NR, AUC 0.28; GSE16879_UC
  0.073 vs 0.182, AUC 0.09; GSE16879_CD 0.040 vs 0.138, AUC 0.09).
  Compositional caveat: fractions are relative - a denser immune/epithelial
  compartment mechanically lowers the fibroblast share, so this is not
  fibroblast loss; it is directionally consistent with the nb09/nb10
  program-score association (higher IAF program in non-responders).
- **GSE100833:** curation template saved; label parsing not wired - under
  the downgrade the primary analysis does not proceed, so the cohort is held.
- **Manuscript:** nb12 adds a bounded negative to the bulk section -
  program scores carry association (nb09/nb10, v2-adjusted); composition
  does not resolve (nb12). Same shape as the blood result (Module 6): the
  signal lives in the compartment where it was defined.
